# Team 04 End-to-End API Notebook

This notebook exercises the current LangGraph planner plus supervisor flow with live LLM API calls while keeping the site and evaluation tools notebook-local and deterministic.

## Scope

This is the right notebook when you want to validate the current supervision pattern end to end:
1. planner chooses the next active step
2. supervisor decides only for the active step
3. tools execute and hand control back to the planner

It avoids a live MCP dependency by using notebook-local site context, legal constraints, and evaluation tools.

If this fails before the agent runs, check the Team 04 Python environment and the `.env` provider settings first.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

workspace_root = Path.cwd().resolve()
candidate_roots = (
    workspace_root,
    workspace_root.parent,
    workspace_root / "team_04",
    workspace_root.parent / "team_04",
)
TEAM_ROOT = next((path for path in candidate_roots if (path / "agent").exists()), None)
if TEAM_ROOT is None:
    raise FileNotFoundError(
        "Run this notebook from the workspace root, the team_04 folder, or the team_04/test_notebooks folder."
    )

team_root_str = str(TEAM_ROOT)
if team_root_str not in sys.path:
    sys.path.insert(0, team_root_str)

E2E_OUTPUT_PATH = TEAM_ROOT / "test_notebooks" / "end_to_end_api_agent_output.json"
E2E_OUTPUT_PATH

WindowsPath('C:/Users/baoqt/OneDrive/Documents/GitHub/AIA26_Studio/team_04/test_notebooks/end_to_end_api_agent_output.json')

In [3]:
from langchain_openai import ChatOpenAI

from agent.config import load_settings
from agent.decision_engine import OpenAIDecisionEngine, RuleBasedPlanner
from agent.graph import run_agent
from agent.mcp_client import CompositeToolClient, build_default_local_tool_client
from agent.notebook_demo_tools import build_notebook_demo_tool_client
from agent.tool_catalog import ToolCatalog

settings_error = None
try:
    settings = load_settings()
except Exception as exc:
    settings = None
    settings_error = str(exc)

SITE_BOUNDARY = [
    [0.0, 0.0, 0.0],
    [90.0, 0.0, 0.0],
    [90.0, 60.0, 0.0],
    [0.0, 60.0, 0.0],
    [0.0, 0.0, 0.0],
]

layout_payload = {
    "workflow_mode": "full",
    "site_boundary": SITE_BOUNDARY,
    "target_building_count": 1,
    "building_intents": ["Keep the wing graph readable for later edits."],
}

prompt = (
    "Place a U-shaped building of 900 square meters inside the site boundary. "
    "Keep the wing structure readable so later agent steps can adjust individual wings."
 )

{
    "llm_provider": settings.llm_provider if settings is not None else None,
    "llm_model": settings.llm_model if settings is not None else None,
    "settings_error": settings_error,
}

{'llm_provider': None,
 'llm_model': None,
 'settings_error': 'Missing required environment variable: LLM_PROVIDER'}

In [ ]:
if settings is None:
    raise RuntimeError(
        "Missing LLM runtime settings. Define LLM_PROVIDER and provider credentials in the repo .env or team_04/.env before running the end-to-end agent cell."
    )

notebook_client = build_notebook_demo_tool_client(
    SITE_BOUNDARY,
    site_summary="Notebook-local site context for end-to-end supervisor validation.",
    setback_m=5.0,
    spatial_intention_score=0.91,
    performance_score=0.88,
    shape_integrity_score=0.94,
)
tool_client = CompositeToolClient([build_default_local_tool_client(), notebook_client])
catalog = ToolCatalog.from_discovered_tools(tool_client.list_tools())

llm = ChatOpenAI(
    api_key=settings.api_key,
    base_url=settings.base_url,
    model=settings.llm_model,
    timeout=settings.request_timeout_seconds,
    temperature=0,
)
decision_engine = OpenAIDecisionEngine(
    llm=llm,
    decision_provider=settings.decision_llm_provider or settings.llm_provider,
    decision_model=settings.decision_llm_model or settings.llm_model,
    report_provider=settings.report_llm_provider or settings.llm_provider,
    report_model=settings.report_llm_model or settings.llm_model,
)

final_state = run_agent(
    user_prompt=prompt,
    decision_engine=decision_engine,
    tool_client=tool_client,
    catalog=catalog,
    initial_layout=layout_payload,
    max_optimization_cycles=2,
    planner=RuleBasedPlanner(),
)

In [ ]:
decision_trace = [
    message
    for message in final_state.get("messages", [])
    if message.startswith(("Planner updated", "Supervisor decision", "Tool ", "Final report"))
]

summary = {
    "final_response": final_state.get("final_response"),
    "geometry_id": final_state.get("geometry_id"),
    "workflow_mode": final_state.get("workflow_mode"),
    "violations": final_state.get("violations"),
    "placed_buildings": final_state.get("placed_buildings"),
    "shape_context_keys": sorted((final_state.get("shape_context") or {}).keys()),
    "decision_trace": decision_trace,
}

E2E_OUTPUT_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary

In [ ]:
{
    "plan": final_state.get("plan"),
    "tool_sequence": [
        record.get("tool")
        for record in final_state.get("tool_history", [])
        if isinstance(record, dict)
    ],
    "saved_to": str(E2E_OUTPUT_PATH),
}